# Deep Agents: Planning, Files, and Subagents

This notebook tests the `deepagents` library with a compact course-assistant use case. It exercises the main capabilities: custom tools, built-in planning, virtual files, subagents, streaming, structured output, and human-in-the-loop interrupts.

You need `OPENAI_API_KEY` in the repository `.env` file or in your shell environment. You can set `OPENAI_MODEL` too; otherwise the notebook uses `openai:gpt-4o-mini`.

## 1. Imports and Configuration

In [1]:
import os
from pprint import pprint
from typing import Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from deepagents import create_deep_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

MODEL_NAME = os.getenv("OPENAI_MODEL", "openai:gpt-4o-mini")
print(f"Using model: {MODEL_NAME}")

c:\Users\A200239740\AppData\Local\miniconda3\envs\agents\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Using model: openai:gpt-4o-mini


## 2. Define Local Tools

The agent can use any LangChain-compatible tool. Here the tools are deterministic and local so the example stays focused on the Deep Agents harness rather than web access.

In [2]:
COURSE_NOTES = {
    "langchain": "LangChain provides model, prompt, tool, message, and runnable abstractions for LLM applications.",
    "langgraph": "LangGraph represents agent workflows as stateful graphs with nodes, edges, conditional routing, checkpointing, and streaming.",
    "react": "ReAct agents alternate reasoning, action, observation, and final answer. Tool calls are selected dynamically by the model.",
    "reflection": "Reflection agents draft, critique, and revise their own output through an explicit feedback loop.",
    "reflexion": "Reflexion agents add memory of prior failures and use that memory to improve future attempts.",
    "agentic rag": "Agentic RAG uses agents to route questions, retrieve context, draft answers, verify support, and retry when evidence is weak.",
    "deep agents": "Deep Agents is an agent harness built on LangChain and LangGraph. It adds planning, files, subagents, and context management defaults.",
}


@tool
def search_course_notes(query: str) -> str:
    """Search concise local notes about LangChain, LangGraph, and agent patterns."""
    query_terms = query.lower().split()
    matches = []
    for key, value in COURSE_NOTES.items():
        haystack = f"{key} {value}".lower()
        if any(term in haystack for term in query_terms):
            matches.append(f"## {key}\n{value}")
    return "\n\n".join(matches) if matches else "No local note matched the query."


@tool
def compare_patterns(left: str, right: str) -> str:
    """Return a compact comparison frame for two agent patterns."""
    return (
        f"Compare {left} and {right} across: control flow, state, context management, "
        "tool use, reliability controls, and best-fit use cases."
    )


@tool
def rate_answer(answer: str) -> str:
    """Rate an answer draft for specificity, evidence, and actionability."""
    checks = {
        "mentions concrete capabilities": any(word in answer.lower() for word in ["planning", "files", "subagent", "interrupt"]),
        "mentions when to use it": "use" in answer.lower() or "best" in answer.lower(),
        "is concise": len(answer.split()) <= 250,
    }
    return "\n".join(f"- {name}: {value}" for name, value in checks.items())


tools = [search_course_notes, compare_patterns, rate_answer]
print([tool.name for tool in tools])

['search_course_notes', 'compare_patterns', 'rate_answer']


## 3. Configure Specialist Subagents

Subagents are specialists with their own prompts, optional models, and optional tool sets. The main agent receives a `task` tool and decides when to delegate.

In [3]:
research_subagent = {
    "name": "course-researcher",
    "description": "Research one narrow LangChain, LangGraph, or Deep Agents topic using local course notes.",
    "system_prompt": (
        "You are a careful course researcher. Search the notes, extract only relevant facts, "
        "and return compact findings with any uncertainty stated clearly."
    ),
    "tools": [search_course_notes],
}

critic_subagent = {
    "name": "answer-critic",
    "description": "Critique a short explanation for missing capabilities, unclear tradeoffs, or unsupported claims.",
    "system_prompt": (
        "You are a concise reviewer. Identify missing Deep Agents capabilities, weak comparisons, "
        "and unclear wording. Prefer bullet points."
    ),
    "tools": [rate_answer],
}

subagents = [research_subagent, critic_subagent]
print([agent["name"] for agent in subagents])

['course-researcher', 'answer-critic']


## 4. Create the Deep Agent

`create_deep_agent(...)` returns a compiled LangGraph app. The built-in tools include todo planning and virtual filesystem operations; custom tools and subagents are added on top.

In [4]:
SYSTEM_PROMPT = """
You are a teaching assistant for an agentic AI course.

For non-trivial questions:
- Use write_todos to plan the work.
- Delegate narrow research tasks to course-researcher.
- Use answer-critic before the final response when you have a draft.
- Use virtual files for durable working notes. Prefer /research_notes.md, /draft.md, and /final_answer.md.
- Keep the final answer concise and practical.
"""

agent = create_deep_agent(
    model=MODEL_NAME,
    tools=tools,
    subagents=subagents,
    system_prompt=SYSTEM_PROMPT,
    name="deepagents-course-assistant",
)

print(type(agent))

<class 'langgraph.graph.state.CompiledStateGraph'>


## 5. Run a Planning + Files + Subagents Task

This prompt asks for behavior that should trigger several capabilities: todo planning, local search, subagent delegation, virtual file writes, critique, and synthesis.

In [5]:
task = (
    "Explain when to use LangChain create_agent, a custom LangGraph workflow, "
    "and Deep Agents. Research the course notes, delegate at least one focused "
    "subtask, write working notes to /research_notes.md, write a draft to /draft.md, "
    "ask the critic to review the draft, then write the final answer to /final_answer.md "
    "before responding."
)

result = agent.invoke({"messages": [{"role": "user", "content": task}]})

print(result["messages"][-1].content)

[{'type': 'text', 'text': "The research on when to use LangChain's `create_agent`, custom LangGraph workflows, and Deep Agents is complete. Here’s a concise overview:\n\n### 1. **create_agent**\n- **Use Cases**:\n  - Dynamic interactivity (e.g., customer service chatbots).\n  - Complex task automation (e.g., travel planners).\n\n- **Practical Examples**:\n  - Customer support agents responding to FAQs.\n  - Interactive learning assistants guiding users.\n\n### 2. **Custom LangGraph Workflows**\n- **Use Cases**:\n  - Stateful process management for complex tasks.\n  - Conditional logic routing based on user inputs.\n\n- **Practical Examples**:\n  - E-commerce recommendation engines suggesting products.\n  - Application review pipelines managing approvals based on specific criteria.\n\n### 3. **Deep Agents**\n- **Use Cases**:\n  - Advanced planning and context management.\n  - Sub-agent capabilities for coherent task management.\n\n- **Practical Examples**:\n  - Project management assist

## 6. Inspect Messages, State, and Virtual Files

The exact returned state may vary by `deepagents` version, but messages and files are the most useful fields to inspect after a run.

In [6]:
print("State keys:", sorted(result.keys()))
print("Message count:", len(result.get("messages", [])))

for index, message in enumerate(result.get("messages", [])[-6:], start=1):
    print(f"\n--- recent message {index}: {message.type} ---")
    content = getattr(message, "content", "")
    print(str(content)[:1200])

files = result.get("files", {})
print("\nFiles:", list(files.keys()))
for path, info in files.items():
    print(f"\n--- {path} ---")
    print(info.get("content", "")[:1500])

State keys: ['files', 'messages']
Message count: 14

--- recent message 1: tool ---
Updated file /draft.md

--- recent message 2: ai ---
[{'arguments': '{"answer":"# Draft: When to Use LangChain create_agent, Custom LangGraph Workflows, and Deep Agents\\n\\n## 1. create_agent\\n### Use Cases:\\n- **Dynamic Interactivity**: Agents can perform tasks interactively, adapting to user input in real-time (e.g., customer service chatbots).\\n- **Complex Task Automation**: Automates workflows that require sequential decision-making (e.g., travel planners).\\n\\n### Practical Examples:\\n- **Customer Support Agent**: Answers FAQs and escalates complex issues.\\n- **Interactive Learning Assistant**: Guides users in navigating complex subjects.\\n\\n## 2. Custom LangGraph Workflows\\n### Use Cases:\\n- **Stateful Process Management**: Manages complex, multi-stage tasks.\\n- **Conditional Logic**: Routes processes based on user input (e.g., personalized business guidance).\\n\\n### Practical Exampl

## 7. Stream the Compiled LangGraph App

Deep Agents are LangGraph-native, so `.stream(...)` works like it does for other compiled graphs.

In [7]:
stream_task = (
    "Create a compact checklist for deciding whether a use case needs Deep Agents. "
    "Save it to /checklist.md and then summarize it."
)

for chunk in agent.stream({"messages": [{"role": "user", "content": stream_task}]}, stream_mode="updates"):
    pprint(chunk)

{'PatchToolCallsMiddleware.before_agent': None}
{'model': {'messages': [AIMessage(content=[{'arguments': '{"file_path":"/checklist.md","content":"# Deep Agents Use Case Checklist\\n\\n1. **Complexity**  \\n   - Is the use case multi-faceted with interdependencies?\\n   - Does it require specialized skills or extensive expertise?\\n\\n2. **Autonomy**  \\n   - Can the task be delegated autonomously?\\n   - Is it valuable to separate from the main workflow?\\n\\n3. **Token Efficiency**  \\n   - Will isolating the task save tokens or reduce context loss?\\n   - Are there computation-heavy tasks that can be offloaded?\\n\\n4. **Parallel Work**  \\n   - Does the use case allow for independent tasks to run simultaneously?\\n   - Is there potential for parallel processing benefits?\\n\\n5. **Outcome Focused**  \\n   - Is the expected output straightforward with little need for intermediate checks?\\n   - Do you care primarily about the end result rather than the process?\\n\\n6. **Complex Requ

## 8. Structured Output

`create_deep_agent(...)` also accepts a response schema. This is useful when the final result must feed another program instead of a human-readable answer.

In [8]:
class FrameworkChoice(BaseModel):
    recommended_framework: Literal["LangChain", "LangGraph", "Deep Agents"] = Field(
        description="The best framework for the described use case."
    )
    reason: str = Field(description="Short reason for the recommendation.")
    required_capabilities: list[str] = Field(description="Capabilities that drive the choice.")


structured_agent = create_deep_agent(
    model=MODEL_NAME,
    tools=[search_course_notes],
    system_prompt="Recommend the smallest LangChain ecosystem tool that fits the user's use case.",
    response_format=FrameworkChoice,
)

structured_result = structured_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "I need a long-running research assistant that writes intermediate notes and delegates focused searches.",
            }
        ]
    }
)

print(structured_result.get("structured_response"))

recommended_framework='LangChain' reason='LangChain is built for orchestrating complex workflows and can manage the entire research process effectively, including delegating focused searches and maintaining intermediate notes.' required_capabilities=['long-running research management', 'intermediate note-taking', 'delegation of focused searches', 'synthesis of results']


## 9. Human-in-the-Loop Interrupts

Use `interrupt_on` when built-in tools should pause for approval. This example interrupts before virtual file writes and edits. Re-run or resume according to the interrupt object returned by your installed LangGraph version.

In [9]:
reviewed_agent = create_deep_agent(
    model=MODEL_NAME,
    tools=[search_course_notes],
    system_prompt="You are a cautious assistant. Ask for approval before changing files.",
    interrupt_on={
        "write_file": True,
        "edit_file": True,
    },
    checkpointer=MemorySaver(),
)

config = {"configurable": {"thread_id": "deepagents-hitl-demo"}}

hitl_result = reviewed_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Write a two-sentence summary of Deep Agents to /brief.md.",
            }
        ]
    },
    config=config,
)

print(hitl_result.keys())
if "__interrupt__" in hitl_result:
    print("Paused for approval:")
    pprint(hitl_result["__interrupt__"])
else:
    print(hitl_result["messages"][-1].content)

dict_keys(['messages', '__interrupt__'])
Paused for approval:
[Interrupt(value={'action_requests': [{'args': {'content': 'Deep Agents are AI '
                                                           'systems designed '
                                                           'to perform complex '
                                                           'tasks '
                                                           'autonomously, '
                                                           'utilizing '
                                                           'subagents to '
                                                           'manage specific '
                                                           'operations in '
                                                           'isolation. This '
                                                           'structure enhances '
                                                           'efficiency and '
                

## 10. What This Notebook Covered

- Custom LangChain tools are passed directly to `create_deep_agent(...)`.
- The main agent can use built-in todo planning for multi-step work.
- Virtual files provide durable working memory for notes, drafts, and final artifacts.
- Subagents isolate specialist research and critique context from the main agent context.
- The compiled app supports normal LangGraph invocation and streaming.
- Pydantic schemas can constrain final output shape.
- `interrupt_on` and `MemorySaver` enable approval and resumability for sensitive operations.